In [1]:
# ============================================================
# AI-POWERED PARAPHRASING TOOL (FINAL CLEAN VERSION)
# Uses: T5 + TextBlob + BLEU + ROUGE + Semantic Similarity
# NO JAVA REQUIRED ❌
# ============================================================

# ── STEP 1: INSTALL LIBRARIES ───────────────────────────────
!pip install transformers sentencepiece torch --quiet
!pip install textblob nltk rouge-score sentence-transformers --quiet

# ── STEP 2: IMPORTS ─────────────────────────────────────────
import textwrap
import nltk
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from textblob import TextBlob
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer
from sentence_transformers import SentenceTransformer, util

nltk.download('punkt')

print("All libraries loaded successfully")

# ── STEP 3: LOAD MODELS ─────────────────────────────────────
MODEL_NAME = "ramsrigouthamg/t5_paraphraser"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

sim_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

print("Models loaded successfully")

# ── STEP 4: PARAPHRASING FUNCTION ───────────────────────────
def paraphrase_text(text, num_return_sequences=1, num_beams=4, max_length=128):

    input_text = f"paraphrase: {text} </s>"

    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        padding=True,
        truncation=True
    )

    outputs = model.generate(
        **inputs,
        max_length=max_length,
        num_beams=num_beams,
        num_return_sequences=num_return_sequences,
        early_stopping=True
    )

    return [
        tokenizer.decode(out, skip_special_tokens=True)
        for out in outputs
    ]

# ── STEP 5: GRAMMAR CHECK (SAFE VERSION - NO JAVA) ───────────
def check_grammar(text):
    corrected = str(TextBlob(text).correct())
    return {
        "corrected_text": corrected,
        "issue_count": 0
    }

# ── STEP 6: METRICS ─────────────────────────────────────────
def compute_bleu(ref, cand):
    smoothie = SmoothingFunction().method4
    ref_tokens = nltk.word_tokenize(ref.lower())
    cand_tokens = nltk.word_tokenize(cand.lower())
    return sentence_bleu([ref_tokens], cand_tokens, smoothing_function=smoothie)


def compute_rouge(ref, cand):
    scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
    return scorer.score(ref, cand)['rougeL'].fmeasure


def compute_similarity(ref, cand):
    emb = sim_model.encode([ref, cand], convert_to_tensor=True)
    return util.cos_sim(emb[0], emb[1]).item()

# ── STEP 7: FULL PIPELINE ───────────────────────────────────
def run_pipeline(text, n=2):

    print("\n" + "="*80)
    print("ORIGINAL TEXT:")
    print(textwrap.fill(text, 80))
    print("="*80)

    paraphrases = paraphrase_text(text, num_return_sequences=n, num_beams=max(4, n))

    for i, p in enumerate(paraphrases, 1):

        print("\n" + "-"*80)
        print(f"PARAPHRASE {i}:")
        print(textwrap.fill(p, 80))

        corrected = check_grammar(p)["corrected_text"]

        print("\nAfter Grammar Correction:")
        print(textwrap.fill(corrected, 80))

        bleu = compute_bleu(text, corrected)
        rouge = compute_rouge(text, corrected)
        sim = compute_similarity(text, corrected)

        print("\nMETRICS:")
        print(f"BLEU  : {bleu:.4f}")
        print(f"ROUGE : {rouge:.4f}")
        print(f"SIM   : {sim:.4f}")

# ── STEP 8: TEST INPUT ──────────────────────────────────────
text = """
Machine learning enables computers to learn from data and make predictions without being explicitly programmed.
"""

run_pipeline(text, n=2)

# ── STEP 9: EXTRA TEST CASES ────────────────────────────────
tests = [
    "Deep learning uses multiple neural network layers to extract features.",
    "Neural networks are inspired by the human brain.",
    "NLP allows machines to understand human language."
]

for i, t in enumerate(tests, 1):
    print("\n" + "#"*80)
    print(f"TEST CASE {i}")
    run_pipeline(t, n=1)

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Jincy\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


All libraries loaded successfully


Loading weights:   0%|          | 0/260 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Models loaded successfully

ORIGINAL TEXT:
 Machine learning enables computers to learn from data and make predictions
without being explicitly programmed.

--------------------------------------------------------------------------------
PARAPHRASE 1:
Machine learning enables computers to learn from data and make predictions
without programming.

After Grammar Correction:
Machine learning enables computers to learn from data and make predictions
without programming.

METRICS:
BLEU  : 0.7416
ROUGE : 0.9286
SIM   : 0.9521

--------------------------------------------------------------------------------
PARAPHRASE 2:
Machine learning enables computers to learn from data without programming.

After Grammar Correction:
Machine learning enables computers to learn from data without programming.

METRICS:
BLEU  : 0.4555
ROUGE : 0.8000
SIM   : 0.9102

################################################################################
TEST CASE 1

ORIGINAL TEXT:
Deep learning uses multiple neural n